In [23]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Retail Analytics") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 3.5.6


In [24]:
customers_df = spark.read.csv("data/customers.csv", header=True, inferSchema=True)

products_df = spark.read.csv("data/products.csv", header=True, inferSchema=True)

orders_df = spark.read.csv("data/orders.csv", header=True, inferSchema=True)

returns_df = spark.read.csv("data/returns.csv", header=True, inferSchema=True)
order_items_df = spark.read.csv("data/order_items.csv", header=True, inferSchema=True)


In [25]:
total_customers = customers_df.count()
total_products = products_df.count()
total_orders = orders_df.count()
total_returned_orders = returns_df.count()

print(f"Total Customers      : {total_customers}")
print(f"Total Products       : {total_products}")
print(f"Total Orders         : {total_orders}")
print(f"Total Returned Orders: {total_returned_orders}")

Total Customers      : 100000
Total Products       : 50000
Total Orders         : 1000000
Total Returned Orders: 100000


In [26]:
q1_result = spark.createDataFrame([
    ("Customers", customers_df.count()),
    ("Products", products_df.count()),
    ("Orders", orders_df.count()),
    ("Returned Orders", returns_df.count())
], ["Metric", "Count"])

q1_result.show()

q1_result.write.mode("overwrite") \
    .option("header", "true") \
    .csv("output/q1_summary_counts")

+---------------+-------+
|         Metric|  Count|
+---------------+-------+
|      Customers| 100000|
|       Products|  50000|
|         Orders|1000000|
|Returned Orders| 100000|
+---------------+-------+



In [27]:
order_items_df.createOrReplaceTempView("order_items")
products_df.createOrReplaceTempView("products")

In [28]:
q2_result = spark.sql("""
SELECT
    p.category,
    SUM(oi.quantity * oi.selling_price) AS total_sales
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_sales DESC
""")

q2_result.show(truncate=False)
q2_result.write.mode("overwrite") \
    .option("header", "true") \
    .csv("output/q2_total_sales_by_category")

+--------------+-------------------+
|category      |total_sales        |
+--------------+-------------------+
|Beauty        |7.626693058999963E8|
|Home & Kitchen|7.581388732799902E8|
|Books         |7.464907783499908E8|
|Toys          |7.446190722999846E8|
|Electronics   |7.442665041099958E8|
|Sports        |7.433388681300008E8|
|Clothing      |7.419227945699946E8|
+--------------+-------------------+



In [29]:
customers_df.createOrReplaceTempView("customers")
orders_df.createOrReplaceTempView("orders")
order_items_df.createOrReplaceTempView("order_items")

In [30]:
top10_customers = spark.sql("""
SELECT
    c.customer_id,
    c.customer_name,
    ROUND(SUM(oi.quantity * oi.selling_price), 2) AS total_purchase_amount
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_id, c.customer_name
ORDER BY total_purchase_amount DESC
LIMIT 10
""")

top10_customers.show()

[Stage 179:============================>                            (1 + 1) / 2]

+-----------+--------------+---------------------+
|customer_id| customer_name|total_purchase_amount|
+-----------+--------------+---------------------+
|      93094|Customer_93094|            181569.68|
|      64560|Customer_64560|             169060.4|
|      23289|Customer_23289|             161573.8|
|      52275|Customer_52275|            153364.79|
|      61218|Customer_61218|            153067.55|
|      52034|Customer_52034|            152680.05|
|      40442|Customer_40442|            151037.32|
|      60528|Customer_60528|            148691.95|
|      84830|Customer_84830|            148363.84|
|      82593|Customer_82593|            148281.04|
+-----------+--------------+---------------------+



In [31]:
top10_customers.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("output/q3_top10_customers")

In [34]:
q4 = spark.sql("""
WITH latest_year AS (
    SELECT MAX(YEAR(order_date)) AS yr
    FROM orders
)

SELECT
    MONTH(o.order_date) AS month,
    ROUND(SUM(oi.quantity * oi.selling_price), 2) AS total_sales
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
CROSS JOIN latest_year ly
WHERE YEAR(o.order_date) = ly.yr
GROUP BY MONTH(o.order_date)
ORDER BY month;
""")

q4.show()
q4.write.mode("overwrite").option("header","true")\
.csv("output/q5_return_percentage")

+-----+--------------+
|month|   total_sales|
+-----+--------------+
|    1|4.4457777576E8|
|    2| 4.153661442E8|
|    3|4.4362824541E8|
|    4|4.2782097434E8|
|    5|4.4481061895E8|
|    6|4.3170515406E8|
|    7|4.4367051912E8|
|    8|4.4109517702E8|
|    9|4.3107152608E8|
|   10|4.4136378931E8|
|   11|4.3362336404E8|
|   12|4.4271290835E8|
+-----+--------------+



In [ ]:
q5 = spark.sql("""
SELECT
    p.category,
    ROUND(
        COUNT(DISTINCT r.order_id)*100.0/
        COUNT(DISTINCT o.order_id),2
    ) AS return_percentage
FROM products p
JOIN order_items oi
ON p.product_id=oi.product_id
JOIN orders o
ON oi.order_id=o.order_id
LEFT JOIN returns r
ON o.order_id=r.order_id
GROUP BY p.category
ORDER BY return_percentage DESC
""")

q5.show()

q5.write.mode("overwrite").option("header","true")\
.csv("output/q5_return_percentage")